# CONFIGURATION

In [ ]:
# Data generation settings
TARGET_CLASS = "var_Cataclysmic"  # ⬅️ CHANGE THIS: Which class to generate
NUM_SAMPLES_TO_GENERATE = 2500   # ⬅️ CHANGE THIS: How many samples to generate

# cVAE Hyperparameters
LATENT_DIM = 10          # Size of latent space
HIDDEN_DIM = 128         # Size of hidden layers
LEARNING_RATE = 1e-3     # Learning rate
BATCH_SIZE = 64          # Batch size
NUM_EPOCHS = 100         # Maximum number of epochs
BETA = 1.0               # Weight for KL divergence (β-VAE)

# Early stopping
PATIENCE = 15            # Stop if no improvement for this many epochs

# Paths
DATA_PATH = '/content/drive/MyDrive/NARIT/Project2_Syntheticdata/Dataset/prepared_data_80_20.pkl'
MODEL_SAVE_PATH = '/content/drive/MyDrive/NARIT/Project2_Syntheticdata/Models/best_cvae_model.pth'
SYNTHETIC_DATA_PATH = '/content/drive/MyDrive/NARIT/Project2_Syntheticdata/Dataset/synthetic_data_single_class_Cat2500.pkl'

# STEP 1: Load Prepared Data

In [ ]:
print("\n📂 Loading prepared data...")
loaded_data = joblib.load(DATA_PATH)

X_train = loaded_data['X_train']
X_val = loaded_data['X_val']
y_train = loaded_data['y_train']
y_val = loaded_data['y_val']
label_encoder = loaded_data['label_encoder']
scaler = loaded_data['scaler']
num_features = loaded_data['num_features']
num_classes = loaded_data['num_classes']

print(f"✓ Data loaded successfully!")
print(f"  Train samples: {X_train.shape[0]}")
print(f"  Val samples: {X_val.shape[0]}")
print(f"  Features: {num_features}")
print(f"  Classes: {num_classes}")
print(f"  Class names: {list(label_encoder.classes_)}")

# Verify target class exists
if TARGET_CLASS not in label_encoder.classes_:
    raise ValueError(f"Target class '{TARGET_CLASS}' not found in classes: {list(label_encoder.classes_)}")

target_class_index = label_encoder.transform([TARGET_CLASS])[0]
print(f"\n🎯 Target class for generation: '{TARGET_CLASS}' (index: {target_class_index})")
print(f"   Generating {NUM_SAMPLES_TO_GENERATE:,} samples")

# STEP 2: Setup Device

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n💻 Using device: {device}")

In [ ]:
if device == "cuda":
  torch.cuda.empty_cache()
  print("✓ GPU cache cleared")
else:
  print("You are using CPU.")

# STEP 3: Create PyTorch Dataset and DataLoader

In [ ]:
class LightCurveDataset(Dataset):
    """Custom dataset for light curve features with labels"""
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Create datasets
train_dataset = LightCurveDataset(X_train, y_train)
val_dataset = LightCurveDataset(X_val, y_val)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n✓ DataLoaders created:")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")

# STEP 4: Define cVAE Architecture

In [ ]:
class Encoder(nn.Module):
    """Encoder: Takes features + condition → outputs mean and log_variance"""
    def __init__(self, input_dim, condition_dim, hidden_dim, latent_dim):
        super(Encoder, self).__init__()

        self.fc1 = nn.Linear(input_dim + condition_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc_mean = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x, c):
        xc = torch.cat([x, c], dim=1)
        h = F.relu(self.fc1(xc))
        h = F.relu(self.fc2(h))
        mean = self.fc_mean(h)
        logvar = self.fc_logvar(h)
        return mean, logvar


class Decoder(nn.Module):
    """Decoder: Takes latent sample + condition → reconstructs features"""
    def __init__(self, latent_dim, condition_dim, hidden_dim, output_dim):
        super(Decoder, self).__init__()

        self.fc1 = nn.Linear(latent_dim + condition_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

    def forward(self, z, c):
        zc = torch.cat([z, c], dim=1)
        h = F.relu(self.fc1(zc))
        h = F.relu(self.fc2(h))
        x_recon = self.fc3(h)
        return x_recon


class cVAE(nn.Module):
    """Conditional Variational Autoencoder"""
    def __init__(self, input_dim, condition_dim, hidden_dim, latent_dim):
        super(cVAE, self).__init__()

        self.encoder = Encoder(input_dim, condition_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, condition_dim, hidden_dim, input_dim)
        self.condition_dim = condition_dim
        self.latent_dim = latent_dim

    def reparameterize(self, mean, logvar):
        """Reparameterization trick: z = mean + std * epsilon"""
        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)
        z = mean + std * epsilon
        return z

    def forward(self, x, c):
        """Forward pass through cVAE"""
        c_onehot = F.one_hot(c, num_classes=self.condition_dim).float()
        mean, logvar = self.encoder(x, c_onehot)
        z = self.reparameterize(mean, logvar)
        x_recon = self.decoder(z, c_onehot)
        return x_recon, mean, logvar

    def generate(self, num_samples, condition, device):
        """Generate new samples for a specific class"""
        self.eval()
        with torch.no_grad():
            z = torch.randn(num_samples, self.latent_dim).to(device)
            c = torch.tensor([condition] * num_samples).to(device)
            c_onehot = F.one_hot(c, num_classes=self.condition_dim).float()
            x_gen = self.decoder(z, c_onehot)
        return x_gen.cpu().numpy()


print(f"\n🏗️ Building cVAE model...")
print(f"  Input features: {num_features}")
print(f"  Condition classes: {num_classes}")
print(f"  Latent dimension: {LATENT_DIM}")
print(f"  Hidden dimension: {HIDDEN_DIM}")

# STEP 5: Define Loss Function

In [ ]:
def cvae_loss(x_recon, x, mean, logvar, beta=1.0):
    """
    cVAE loss = Reconstruction Loss + β * KL Divergence
    Returns: total_loss, recon_loss, kl_loss
    """
    recon_loss = F.mse_loss(x_recon, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mean.pow(2) - logvar.exp())
    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss

# STEP 6: Initialize Model and Optimizer

In [ ]:
model = cVAE(
    input_dim=num_features,
    condition_dim=num_classes,
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

total_params = sum(p.numel() for p in model.parameters())
print(f"\n✓ Model initialized")
print(f"  Total parameters: {total_params:,}")
print(f"  Optimizer: Adam (lr={LEARNING_RATE})")

# STEP 7: Training Loop with Metrics Collection

In [ ]:
print("🚀 STARTING TRAINING")
print("="*80)
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Early stopping patience: {PATIENCE}")
print(f"Beta (KL weight): {BETA}")
print("="*80)

# Training metrics storage
training_metrics = {
    'best_epoch': 0,
    'best_val_loss': float('inf'),
    'best_train_total_loss': 0,
    'best_train_recon_loss': 0,
    'best_train_kl_loss': 0,
    'best_val_total_loss': 0,
    'best_val_recon_loss': 0,
    'best_val_kl_loss': 0,
    'training_time': 0,
    'total_epochs_trained': 0
}

best_val_loss = float('inf')
patience_counter = 0
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    # ========================================================================
    # Training Phase
    # ========================================================================
    model.train()
    train_loss_total = 0
    train_recon_total = 0
    train_kl_total = 0

    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [Train]', leave=False)
    for batch_features, batch_labels in train_pbar:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # Forward pass
        x_recon, mean, logvar = model(batch_features, batch_labels)
        loss, recon_loss, kl_loss = cvae_loss(x_recon, batch_features, mean, logvar, BETA)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accumulate losses
        train_loss_total += loss.item()
        train_recon_total += recon_loss.item()
        train_kl_total += kl_loss.item()

        train_pbar.set_postfix({
            'loss': f'{loss.item()/len(batch_features):.4f}'
        })

    # Average training losses
    train_loss_avg = train_loss_total / len(train_dataset)
    train_recon_avg = train_recon_total / len(train_dataset)
    train_kl_avg = train_kl_total / len(train_dataset)

    # ========================================================================
    # Validation Phase
    # ========================================================================
    model.eval()
    val_loss_total = 0
    val_recon_total = 0
    val_kl_total = 0

    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [Val]  ', leave=False)
        for batch_features, batch_labels in val_pbar:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            x_recon, mean, logvar = model(batch_features, batch_labels)
            loss, recon_loss, kl_loss = cvae_loss(x_recon, batch_features, mean, logvar, BETA)

            val_loss_total += loss.item()
            val_recon_total += recon_loss.item()
            val_kl_total += kl_loss.item()

    # Average validation losses
    val_loss_avg = val_loss_total / len(val_dataset)
    val_recon_avg = val_recon_total / len(val_dataset)
    val_kl_avg = val_kl_total / len(val_dataset)

    # ========================================================================
    # Print Progress
    # ========================================================================
    if epoch % 5 == 0 or epoch == 1:
        print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
        print(f"  Train - Total: {train_loss_avg:.4f}, Recon: {train_recon_avg:.4f}, KL: {train_kl_avg:.4f}")
        print(f"  Val   - Total: {val_loss_avg:.4f}, Recon: {val_recon_avg:.4f}, KL: {val_kl_avg:.4f}")

    # ========================================================================
    # Save Best Model and Update Metrics
    # ========================================================================
    if val_loss_avg < best_val_loss:
        best_val_loss = val_loss_avg
        patience_counter = 0

        # Update best metrics
        training_metrics['best_epoch'] = epoch
        training_metrics['best_val_loss'] = val_loss_avg
        training_metrics['best_train_total_loss'] = train_loss_avg
        training_metrics['best_train_recon_loss'] = train_recon_avg
        training_metrics['best_train_kl_loss'] = train_kl_avg
        training_metrics['best_val_total_loss'] = val_loss_avg
        training_metrics['best_val_recon_loss'] = val_recon_avg
        training_metrics['best_val_kl_loss'] = val_kl_avg

        # Save model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss_avg,
            'train_loss': train_loss_avg,
            'metrics': training_metrics
        }, MODEL_SAVE_PATH)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  ✓ Best model saved! (Val loss improved: {val_loss_avg:.4f})")
    else:
        patience_counter += 1

    # ========================================================================
    # Early Stopping
    # ========================================================================
    if patience_counter >= PATIENCE:
        print(f"\n⏹️ Early stopping at epoch {epoch}")
        print(f"   No improvement for {PATIENCE} epochs")
        break

# Calculate total training time
training_time = time.time() - start_time
training_metrics['training_time'] = training_time
training_metrics['total_epochs_trained'] = epoch

print(f"\n" + "="*80)
print("✅ TRAINING COMPLETE!")
print("="*80)
print(f"Training time: {training_time/60:.2f} minutes")
print(f"Total epochs: {epoch}")
print(f"Best epoch: {training_metrics['best_epoch']}")
print(f"\nBest Epoch Metrics (Epoch {training_metrics['best_epoch']}):")
print(f"  Train Loss - Total: {training_metrics['best_train_total_loss']:.4f}, "
      f"Recon: {training_metrics['best_train_recon_loss']:.4f}, "
      f"KL: {training_metrics['best_train_kl_loss']:.4f}")
print(f"  Val Loss   - Total: {training_metrics['best_val_total_loss']:.4f}, "
      f"Recon: {training_metrics['best_val_recon_loss']:.4f}, "
      f"KL: {training_metrics['best_val_kl_loss']:.4f}")
print("="*80)

# STEP 8: Load Best Model for Generation

In [ ]:
print(f"\n📥 Loading best model for generation...")
checkpoint = torch.load(MODEL_SAVE_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✓ Best model loaded (from epoch {checkpoint['epoch']})")

# STEP 9: Generate Synthetic Data for Target Class

In [ ]:
print(f"🎨 GENERATING SYNTHETIC DATA")
print("="*80)
print(f"Target class: '{TARGET_CLASS}' (index: {target_class_index})")
print(f"Number of samples: {NUM_SAMPLES_TO_GENERATE:,}")
print("="*80)

generation_start = time.time()

# Generate data
synthetic_features_scaled = model.generate(
    num_samples=NUM_SAMPLES_TO_GENERATE,
    condition=target_class_index,
    device=device
)

generation_time = time.time() - generation_start

print(f"\n✓ Generated {NUM_SAMPLES_TO_GENERATE:,} samples in {generation_time:.2f} seconds")
print(f"  Shape: {synthetic_features_scaled.shape}")
print(f"  Data is in STANDARDIZED form (same scale as training data)")

# Create labels for synthetic data
synthetic_labels = np.array([target_class_index] * NUM_SAMPLES_TO_GENERATE)

# STEP 10: Save Generated Data

In [ ]:
print(f"\n💾 Saving synthetic data...")

# Package synthetic data
synthetic_data_package = {
    'features_scaled': synthetic_features_scaled,  # Standardized features
    'labels_encoded': synthetic_labels,            # Encoded labels (integers)
    'target_class': TARGET_CLASS,                  # Class name
    'target_class_index': target_class_index,      # Class index
    'num_samples': NUM_SAMPLES_TO_GENERATE,
    'feature_names': loaded_data.get('feature_names', None),
    'scaler': scaler,  # Include scaler for potential inverse transform
    'label_encoder': label_encoder,
    'generation_time': generation_time,
    'training_metrics': training_metrics,
    'model_config': {
        'latent_dim': LATENT_DIM,
        'hidden_dim': HIDDEN_DIM,
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'beta': BETA
    }
}

# Save to pickle file
joblib.dump(synthetic_data_package, SYNTHETIC_DATA_PATH)

print(f"✓ Synthetic data saved to: {SYNTHETIC_DATA_PATH}")

# STEP 11: Summary and Statistics

In [ ]:
print("📊 SUMMARY")
print("="*80)

print(f"\n🏋️ Training Summary:")
print(f"  Total training time: {training_time/60:.2f} minutes")
print(f"  Epochs trained: {training_metrics['total_epochs_trained']}")
print(f"  Best epoch: {training_metrics['best_epoch']}")
print(f"  Best validation loss: {training_metrics['best_val_loss']:.4f}")

print(f"\n🎨 Generation Summary:")
print(f"  Target class: {TARGET_CLASS}")
print(f"  Samples generated: {NUM_SAMPLES_TO_GENERATE:,}")
print(f"  Generation time: {generation_time:.2f} seconds")
print(f"  Data format: Standardized (same scale as training data)")

print(f"\n📁 Saved Files:")
print(f"  Model: {MODEL_SAVE_PATH}")
print(f"  Synthetic data: {SYNTHETIC_DATA_PATH}")

print(f"\n✅ Synthetic data is ready for classification!")
print(f"   You can load it and combine with real data for XGBoost training.")
print(f"   The data is already STANDARDIZED - same as your training data.")

print("\n" + "="*80)
print("🎉 PROCESS COMPLETE!")

# STEP 12: Display Sample Statistics

In [ ]:
print(f"\n📊 Generated Data Statistics:")
print(f"  Shape: {synthetic_features_scaled.shape}")
print(f"  Mean: {synthetic_features_scaled.mean():.6f}")
print(f"  Std: {synthetic_features_scaled.std():.6f}")
print(f"  Min: {synthetic_features_scaled.min():.6f}")
print(f"  Max: {synthetic_features_scaled.max():.6f}")

print(f"\nNote: Data is in standardized form (scaled).")
print(f"To get original scale, use: scaler.inverse_transform(synthetic_features_scaled)")